In [1]:
# Importing required libraries and initializing environment

# importing core libraries for data processing and deep learning
import numpy as np                 # importing for handling numerical arrays
import pandas as pd               # importing for loading and analyzing tabular data
import random                     # importing for generating random values
import string                     # importing for string manipulation utilities

# importing PyTorch tools for model building and optimization
import torch                      # importing core PyTorch library
import torch.nn as nn            # importing for defining neural network layers
import torch.optim as optim      # importing optimizers like Adam
from torch.utils.data import DataLoader  # importing for creating mini-batches for training

# importing HuggingFace Transformers for text embeddings and BERT models
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder

# importing evaluation metric
from sklearn.metrics import roc_auc_score  # importing for measuring AUC performance

# initializing the device (using GPU if available, else fallback to CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [5]:
# Loading the dataset files and inspecting structure

from google.colab import files

uploaded = files.upload()

# importing pandas
import pandas as pd

# reading the training essays CSV file
# loading train_essays.csv from uploaded path
train_essays = pd.read_csv('train_essays.csv')  # loading training essays

# reading the training prompts CSV file
# loading train_prompts.csv from uploaded path
train_prompts = pd.read_csv('train_prompts.csv')  # loading training prompts

# reading the test essays CSV file
# loading test_essays.csv from uploaded path
test_essays = pd.read_csv('test_essays.csv')  # loading test essays

Saving test_essays.csv to test_essays (2).csv
Saving train_essays.csv to train_essays (2).csv
Saving train_prompts.csv to train_prompts (2).csv


In [6]:
print("Train Essays:")
print(train_essays.head())  # printing first 5 rows from training essays

print("\nTrain Prompts:")
print(train_prompts.head())  # printing first 5 rows from training prompts

print("\nTest Essays:")
print(test_essays.head())  # printing first 5 rows from test essays

Train Essays:
         id  prompt_id                                               text  \
0  0059830c          0  Cars. Cars have been around since they became ...   
1  005db917          0  Transportation is a large necessity in most co...   
2  008f63e3          0  "America's love affair with it's vehicles seem...   
3  00940276          0  How often do you ride in a car? Do you drive a...   
4  00c39458          0  Cars are a wonderful thing. They are perhaps o...   

   generated  
0          0  
1          0  
2          0  
3          0  
4          0  

Train Prompts:
   prompt_id                       prompt_name  \
0          0                   Car-free cities   
1          1  Does the electoral college work?   

                                        instructions  \
0  Write an explanatory essay to inform fellow ci...   
1  Write a letter to your state senator in which ...   

                                         source_text  
0  # In German Suburb, Life Goes On Withou

In [8]:
# column names and number of samples
print("\nTrain Essays shape:", train_essays.shape)  # number of rows and columns
print("Train Prompts shape:", train_prompts.shape)
print("Test Essays shape:", test_essays.shape)

# checking for missing values in training datasets
print("\nMissing values in train_essays:")
print(train_essays.isnull().sum())  # summarizing missing values column-wise

print("\nMissing values in train_prompts:")
print(train_prompts.isnull().sum())

print("\nMissing values in test_essays:")
print(test_essays.isnull().sum())


Train Essays shape: (1378, 4)
Train Prompts shape: (2, 4)
Test Essays shape: (3, 3)

Missing values in train_essays:
id           0
prompt_id    0
text         0
generated    0
dtype: int64

Missing values in train_prompts:
prompt_id       0
prompt_name     0
instructions    0
source_text     0
dtype: int64

Missing values in test_essays:
id           0
prompt_id    0
text         0
dtype: int64


In [9]:
# 3. Preparing the Model

# importing BERT tokenizer and model classes from HuggingFace transformers
from transformers import BertTokenizer, BertModel, BertConfig  # importing for text tokenization and BERT embeddings

# initializing the tokenizer using 'bert-base-uncased' model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # loading BERT tokenizer (lowercase version)

# initializing BERT model for extracting embeddings (not classification head)
embedding_model = BertModel.from_pretrained('bert-base-uncased')  # loading pre-trained BERT base model

# initializing configuration object to reuse later for custom BERT layers
bert_config = BertConfig.from_pretrained('bert-base-uncased')  # loading model configuration (e.g., hidden size, layers)

# moving model to device (GPU or CPU)
embedding_model = embedding_model.to(device)  # sending embedding model to computation device

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [11]:
# 4. Setting Hyperparameters

# defining training and test batch sizes
train_batch_size = 5  # setting mini-batch size for training
test_batch_size = 10   # setting mini-batch size for evaluation (can be larger)

# defining learning rate for optimizers
lr = 0.0002  # initializing learning rate (small value for stable training)

# defining Adam optimizer beta parameters
beta1 = 0.5  # initializing beta1 for momentum term in Adam optimizer
beta2 = 0.999  # initializing beta2 for RMSProp-like term in Adam optimizer

# defining latent dimension of noise vector for generator
nz = 100  # initializing size of noise vector input to the generator

# defining number of training epochs
num_epochs = 10  # initializing total number of training iterations over dataset

# defining number of hidden layers for BERT encoder in our GAN
num_hidden_layers = 6  # initializing how many BERT layers to use in generator/discriminator

# defining ratio of training to testing split
train_ratio = 0.8  # initializing ratio of training data (80% train, 20% test)

In [12]:
# 5. Preparing the Data for Training using Dataset class and DataLoader in PyTorch

# importing PyTorch utilities for data handling (if not already imported)
import torch
from torch.utils.data import Dataset, DataLoader  # importing for dataset and batch loading

# defining a custom Dataset class to hold text and label pairs
class GANDAIGDataset(Dataset):
    # initializing with lists of texts and labels
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    # returning number of samples
    def __len__(self):
        return len(self.texts)

    # returning one text-label pair per index
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

In [14]:
# Checking actual column names in train_prompts
print("Column names in train_prompts:")
print(train_prompts.columns.tolist())  # displaying column names to find the correct name for prompt text

Column names in train_prompts:
['prompt_id', 'prompt_name', 'instructions', 'source_text']


In [16]:
# Checking actual column names in train_essays
print("Column names in train_essays:")
print(train_essays.columns.tolist())

Column names in train_essays:
['id', 'prompt_id', 'text', 'generated']


In [17]:
# combining train_essays with train_prompts
# merging prompt texts to include context
merged_data = pd.merge(train_essays, train_prompts, on="prompt_id")  # joining prompt text with essay text

# creating unified text input by combining prompt text + instructions + essay text
merged_data['full_text'] = merged_data['source_text'] + " " + merged_data['instructions'] + " " + merged_data['text']

# extracting texts and binary labels
texts = merged_data['full_text'].tolist()  # collecting full text data
labels = merged_data['generated'].tolist()  # collecting target labels (0 = human-written, 1 = AI-generated)

In [20]:
print(merged_data.head())  # preview the merged dataset

         id  prompt_id                                               text  \
0  0059830c          0  Cars. Cars have been around since they became ...   
1  005db917          0  Transportation is a large necessity in most co...   
2  008f63e3          0  "America's love affair with it's vehicles seem...   
3  00940276          0  How often do you ride in a car? Do you drive a...   
4  00c39458          0  Cars are a wonderful thing. They are perhaps o...   

   generated      prompt_name  \
0          0  Car-free cities   
1          0  Car-free cities   
2          0  Car-free cities   
3          0  Car-free cities   
4          0  Car-free cities   

                                        instructions  \
0  Write an explanatory essay to inform fellow ci...   
1  Write an explanatory essay to inform fellow ci...   
2  Write an explanatory essay to inform fellow ci...   
3  Write an explanatory essay to inform fellow ci...   
4  Write an explanatory essay to inform fellow ci...   

 

In [21]:
# splitting data into training and testing based on ratio
train_size = int(train_ratio * len(texts))  # calculating number of training samples
test_size = len(texts) - train_size  # calculating remaining samples for test set

train_texts = texts[:train_size]  # slicing training texts
train_labels = labels[:train_size]  # slicing training labels

test_texts = texts[train_size:]  # slicing test texts
test_labels = labels[train_size:]  # slicing test labels

In [22]:
# creating Dataset objects
train_dataset = GANDAIGDataset(train_texts, train_labels)  # creating training dataset
test_dataset = GANDAIGDataset(test_texts, test_labels)  # creating testing dataset

# creating DataLoader objects for batching
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)  # batching training data
test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)  # batching test data (no shuffle)

In [24]:
# 6. Defining the Generator Model

# initializing BERT configuration for generator's internal encoder

# setting up BERT encoder config with a custom number of hidden layers
config = BertConfig(num_hidden_layers=num_hidden_layers)

In [25]:
# starting to define our generator class

class Generator(nn.Module):
    def __init__(self, input_dim):
        super(Generator, self).__init__()

        # initializing a linear layer to project latent vector into a higher-dimensional space
        self.fc = nn.Linear(input_dim, 256 * 128)  # (nz → 256 x 128 shape)

        # storing reshape dimensions to use after linear projection
        self.reshape_dim = (256, 128)  # (channels, sequence length)

In [55]:
# Adding ConvTranspose1D layers to upsample latent noise
# Defining the Generator model properly (all parts 1–4)

class Generator(nn.Module):
    def __init__(self, input_dim):
        super(Generator, self).__init__()

        # initializing fully connected layer to expand latent vector
        self.fc = nn.Linear(input_dim, 256 * 128)  # latent → feature map

        # reshaping configuration for conv input
        self.reshape_dim = (256, 128)  # reshape to (batch, channels, seq_len)

        # initializing convolutional upsampling layers
        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.ConvTranspose1d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.ConvTranspose1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        # final projection to 768 (BERT embedding size)
        self.final_linear = nn.Linear(128, 768)

        # BERT encoder
        config = BertConfig(num_hidden_layers=num_hidden_layers)
        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        # expanding latent vector
        x = self.fc(x)  # shape: (B, 256*128)

        # reshaping into 3D for conv layers
        x = x.view(-1, *self.reshape_dim)  # shape: (B, 256, 128)

        # passing through conv layers
        x = self.conv_net(x)  # shape: (B, 128, seq_len)

        # permuting to (B, seq_len, embedding_dim_input)
        x = x.permute(0, 2, 1)  # shape: (B, seq_len, 128)

        # final projection to 768 dims per token
        x = self.final_linear(x)  # shape: (B, seq_len, 768)

        # pass through BERT encoder
        x = self.bert_encoder(x)

        return x.last_hidden_state

In [56]:
# Step 7 - defining a simple embedding pooling mechanism (sum pooling)

class SumBertPooler(nn.Module):
    def __init__(self):
        super(SumBertPooler, self).__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # summing across sequence length dimension
        sum_hidden = hidden_states.sum(dim=1)  # shape → (batch_size, hidden_dim)

        # normalizing the sum by total token values
        sum_mask = sum_hidden.sum(dim=1).unsqueeze(1)  # shape → (batch_size, 1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)  # preventing division by zero

        # returning mean-like pooled embeddings
        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings

In [57]:
# Loading pre-trained BERT model before Discriminator

# loading BERT tokenizer and pre-trained model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased")

# extracting embedding model from pre-trained BERT
embedding_model = pretrained_model.bert

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [58]:
# defining the Discriminator model

# class Discriminator(nn.Module):
#     def __init__(self):
#         super(Discriminator, self).__init__()

#         # initializing BERT encoder (we reuse layers from a pre-trained model)
#         self.bert_encoder = BertEncoder(config)

#         # selecting first few layers from a pre-trained BERT model
#         self.bert_encoder.layer = nn.ModuleList([
#             layer for layer in pretrained_model.bert.encoder.layer[:6]
#         ])

#         # initializing custom pooling method
#         self.pooler = SumBertPooler()

#         # defining a simple classification head (fully connected layers)
#         self.classifier = nn.Sequential(
#             nn.Linear(config.hidden_size, 128),  # input: BERT embedding
#             nn.ReLU(),                           # applying non-linearity
#             nn.Dropout(0.2),                      # preventing overfitting
#             nn.Linear(128, 1)                     # output: one probability score
#         )

#     def forward(self, x):
#         # passing embeddings through the BERT encoder
#         out = self.bert_encoder(x)

#         # pooling encoder output into a single vector
#         pooled_output = self.pooler(out.last_hidden_state)

#         # passing through classification head to get logits
#         out = self.classifier(pooled_output)

#         # applying sigmoid to get probability (real or fake)
#         return torch.sigmoid(out).view(-1)

In [59]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = nn.Sequential(
            nn.Linear(768, 128),  # fixing input dimension here
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, input):
        out = self.bert_encoder(input)
        out = self.pooler(out.last_hidden_state)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)


In [60]:
# preparing BERT embeddings from raw text input

def preparation_embedding(texts):
    # tokenizing texts for BERT
    encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

    # moving tensors to device
    input_ids = encodings['input_ids'].to(device)
    token_type_ids = encodings['token_type_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    # getting embeddings from pretrained BERT
    outputs = embedding_model(
        input_ids=input_ids,
        token_type_ids=token_type_ids,
        attention_mask=attention_mask
    )

    return outputs.last_hidden_state

In [61]:
# 8. Training the Model

# Initializing GAN components before training (Generator, Discriminator, loss function, and optimizers)

# initializing generator and discriminator
netG = Generator(nz).to(device)
netD = Discriminator().to(device)

# setting binary cross entropy as loss function
criterion = nn.BCELoss()

# initializing optimizers for both networks
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

In [64]:
# Writing the GAN training step logic

# defining a single training step for GAN (one mini-batch)

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    # zeroing gradients for discriminator
    netD.zero_grad()
    batch_size = real_data.size(0)

    # training discriminator on real data
    output = netD(real_data)
    errD_real = criterion(output, label)
    errD_real.backward()
    D_x = output.mean().item()

    # generating fake data from random noise
    noise = torch.randn(batch_size, nz, device=device)
    # fake_data = netG(noise).last_hidden_state
    fake_data = netG(noise)

    # training discriminator on fake data
    label.fill_(1)  # fake labels set to 1 (adversarial loss)
    output = netD(fake_data.detach())
    errD_fake = criterion(output, label)
    errD_fake.backward()
    D_G_z1 = output.mean().item()

    # total discriminator loss
    errD = errD_real + errD_fake
    optimizerD.step()

    # training generator to fool discriminator
    netG.zero_grad()
    label.fill_(0)  # generator wants discriminator to think fakes are real
    output = netD(fake_data)
    errG = criterion(output, label)
    errG.backward()
    D_G_z2 = output.mean().item()
    optimizerG.step()

    # printing training progress
    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f' % (
            epoch, num_epochs, i, len(train_loader),
            errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD

In [ ]:
# full training loop over all epochs

model_infos = []

for epoch in range(num_epochs):
    for i, data in enumerate(train_loader, 0):
        # extracting texts and labels from batch
        texts, labels = data

        # getting BERT embeddings for input texts
        with torch.no_grad():
            embeded = preparation_embedding(texts)

        # performing one training step for generator and discriminator
        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=labels.float().to(device),
            epoch=epoch,
            i=i
        )

    # evaluating model on test set using AUC score
    auc_score = eval_auc(netD)

    # saving current model performance
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Training complete!')

[0/10][0/221] Loss_D: 120.0000 Loss_G: 0.0000 D(x): 0.2000 D(G(z)): 0.0000 / 0.0000
[0/10][50/221] Loss_D: 100.0000 Loss_G: 40.0000 D(x): 0.2000 D(G(z)): 0.2000 / 0.4000
